# Session 5: Data Transformation — Subsetting & Feature Engineering
**Python for Data Analysis Workshop | World Development Indicators**

---

## Learning Objectives

By the end of this session you will be able to:
- Select specific columns from a DataFrame
- Filter rows using single and multiple conditions
- Use `.loc[]` and `.iloc[]` for label- and position-based access
- Create new columns (feature engineering)
- Use `apply()` and lambda functions for custom transformations
- Rename and reorder columns

---

## 1. Load the Clean Data

We use the clean dataset produced in Session 4.
If you skipped Session 4, the raw file also works — just note there will be some nulls.

In [ ]:
import pandas as pd
import numpy as np

# Try the clean file first; fall back to raw if not found
try:
    df = pd.read_csv("WDI_clean.csv")
    print("Using clean dataset")
except FileNotFoundError:
    df = pd.read_csv("World_Dev_Indicators.csv")
    df["Region"] = df["Region"].str.strip()
    print("Using raw dataset (with minor fixes)")

print(f"Shape: {df.shape}")
df.head(3)

## 2. Selecting Columns

Choose only the columns relevant to your analysis.
This makes your code clearer and your DataFrames smaller.

In [ ]:
# Select a single column — returns a Series
countries = df["Country"]
print(type(countries))
print(countries.head())

In [ ]:
# Select multiple columns — returns a DataFrame (double brackets)
cols = ["Year","Country","Region","Income Group","GDP per capita (current US$)"]
df_select = df[cols]
print(f"Shape with selected columns: {df_select.shape}")
df_select.head(5)

In [ ]:
# Use .filter() to select columns by partial name match
gdp_cols = df.filter(like="GDP")
print("GDP-related columns:")
print(gdp_cols.columns.tolist())
gdp_cols.head(3)

## 3. Filtering Rows

Filtering rows is one of the most frequent tasks in data analysis.
We use **boolean masks** — conditions that evaluate to True or False for each row.

### 3.1 Single Condition

In [ ]:
# Filter for one specific year
df_2020 = df[df["Year"] == 2020]
print(f"Rows in 2020: {len(df_2020):,}")
df_2020.head(3)

In [ ]:
# Filter for High income countries
df_high = df[df["Income Group"] == "High income"]
print(f"High income country-year observations: {len(df_high):,}")
print(f"Unique countries: {df_high['Country'].nunique()}")

### 3.2 Multiple Conditions

Combine conditions with `&` (AND) and `|` (OR).
Always wrap each condition in parentheses.

In [ ]:
# AND condition — High income AND year 2020
df_high_2020 = df[(df["Income Group"] == "High income") & (df["Year"] == 2020)]
print(f"High income in 2020: {len(df_high_2020)} countries")
df_high_2020.sort_values("GDP per capita (current US$)", ascending=False).head(8)

In [ ]:
# OR condition — South Asia OR Sub-Saharan Africa
df_developing = df[
    (df["Region"] == "South Asia") |
    (df["Region"] == "Sub-Saharan Africa")
]
print(f"South Asia OR Sub-Saharan Africa rows: {len(df_developing):,}")

In [ ]:
# .isin() — filter for multiple specific values at once
selected_countries = ["United States", "China", "India", "Germany", "Brazil"]
df_g5 = df[df["Country"].isin(selected_countries)]
print(f"G5-like country rows: {len(df_g5)}")
print("Countries found:", df_g5["Country"].unique())

In [ ]:
# NOT filter — exclude a group using ~
df_not_high = df[~(df["Income Group"] == "High income")]
print(f"Non-high income rows: {len(df_not_high):,}")

## 4. Label and Position-Based Selection with `.loc[]` and `.iloc[]`

### 4.1 `.loc[]` — Select by Labels

In [ ]:
# .loc[row_labels, column_labels]
# Select rows 0 to 4, specific columns
df.loc[0:4, ["Country","Year","GDP per capita (current US$)"]]

In [ ]:
# .loc with a boolean condition + column selection
df.loc[
    df["Income Group"] == "Low income",
    ["Country","Year","Life expectancy at birth, total (years)"]
].head(8)

### 4.2 `.iloc[]` — Select by Integer Position

In [ ]:
# .iloc[row_positions, column_positions]
# First 5 rows, first 4 columns (by position, not name)
df.iloc[0:5, 0:4]

In [ ]:
# Last 3 rows, last 3 columns
df.iloc[-3:, -3:]

## 5. Feature Engineering — Creating New Columns

Feature engineering means creating new, meaningful columns from existing ones.
This is where analysis creativity shines.

In [ ]:
# 1. GDP in trillions (easier to read than billions)
df["GDP (trillion US$)"] = df["GDP (current US$)"] / 1e12
df[["Country","Year","GDP (current US$)","GDP (trillion US$)"]].head(5)

In [ ]:
# 2. Life expectancy decade bin
df["Life Expectancy Bin"] = pd.cut(
    df["Life expectancy at birth, total (years)"],
    bins=[0, 50, 60, 70, 80, 100],
    labels=["<50","50-60","60-70","70-80","80+"]
)
print("Life expectancy bins:")
print(df["Life Expectancy Bin"].value_counts().sort_index())

In [ ]:
# 3. Decade column from Year
df["Decade"] = (df["Year"] // 10) * 10
print("Decade values:")
print(df["Decade"].value_counts().sort_index())

In [ ]:
# 4. GDP per capita category
def gdp_category(value):
    if pd.isna(value):
        return "Unknown"
    elif value < 1000:
        return "Very Low (<$1k)"
    elif value < 5000:
        return "Low ($1k-$5k)"
    elif value < 20000:
        return "Middle ($5k-$20k)"
    elif value < 50000:
        return "High ($20k-$50k)"
    else:
        return "Very High (>$50k)"

df["GDP per capita Category"] = df["GDP per capita (current US$)"].apply(gdp_category)
print("GDP per capita category distribution:")
print(df["GDP per capita Category"].value_counts())

## 6. Using `apply()` and Lambda Functions

`apply()` runs a function on every row or column.
`lambda` creates a quick, one-line function without `def`.

In [ ]:
# lambda: compute GDP per capita in thousands (one-liner)
df["GDP per capita (k$)"] = df["GDP per capita (current US$)"].apply(
    lambda x: round(x / 1000, 2) if pd.notna(x) else np.nan
)
df[["Country","Year","GDP per capita (current US$)","GDP per capita (k$)"]].head(5)

In [ ]:
# apply() along rows (axis=1) to access multiple columns at once
def classify_development(row):
    """Classify a country-year as developed/developing based on two criteria."""
    gdppc = row["GDP per capita (current US$)"]
    le    = row["Life expectancy at birth, total (years)"]
    if pd.isna(gdppc) or pd.isna(le):
        return "Unknown"
    if gdppc > 15000 and le > 72:
        return "High Development"
    elif gdppc > 5000 or le > 65:
        return "Medium Development"
    else:
        return "Low Development"

df["Development Class"] = df.apply(classify_development, axis=1)
print("Development classification:")
print(df["Development Class"].value_counts())

## 7. Renaming and Reordering Columns

In [ ]:
# Rename specific columns for brevity
df_renamed = df.rename(columns={
    "GDP (current US$)":                         "GDP_USD",
    "GDP per capita (current US$)":              "GDPpc_USD",
    "Life expectancy at birth, total (years)":   "LifeExp",
    "Population, total":                         "Population",
    "Suicide mortality rate (per 100,000 population)": "SuicideRate"
})
print("Renamed columns:")
print(df_renamed.columns.tolist())

In [ ]:
# Reorder columns — put identifiers first, then metrics
col_order = ["Year","Country","Country Code","Region","Income Group",
             "GDP_USD","GDPpc_USD","LifeExp","Population","SuicideRate"]
df_ordered = df_renamed[col_order]
df_ordered.head(3)

## 8. Summary Cheat Sheet

| Task | Code |
|------|------|
| Select columns | `df[['col1','col2']]` |
| Filter one condition | `df[df['col'] == value]` |
| Filter multiple (AND) | `df[(cond1) & (cond2)]` |
| Filter multiple (OR) | `df[(cond1) \| (cond2)]` |
| Filter list of values | `df[df['col'].isin(list)]` |
| Exclude a condition | `df[~condition]` |
| Label-based select | `df.loc[rows, cols]` |
| Position-based select | `df.iloc[row_idx, col_idx]` |
| New column | `df['new'] = expression` |
| Binning | `pd.cut(s, bins=[], labels=[])` |
| Apply function | `df['col'].apply(func)` |
| Row-wise apply | `df.apply(func, axis=1)` |
| Rename columns | `df.rename(columns={'old':'new'})` |

---

## 9. Exercises

1. Create a DataFrame with only Sub-Saharan Africa data from years 2015 to 2025.
2. Create a new column `Population_M` that expresses population in millions (rounded to 2 decimal places).
3. Filter for countries where `Life expectancy at birth` is below 60 **and** the year is 2020.
4. **Bonus:** Write a lambda that returns `'Growing'` if GDP per capita increased from 2019 to 2020 for a country, and `'Shrinking'` if it decreased.

In [ ]:
# Your answers here:
# Exercise 1

# Exercise 2

# Exercise 3

# Exercise 4 (bonus)
